In [5]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
import matplotlib.pyplot as plt

# 1. Constantes Físicas (Valores consolidados para Y2O3)
k54 = 1279.0  # s^-1
k31 = 192.0   # s^-1
k21 = 16.66   # s^-1
sigma = 1.75e-20 # cm^2
P = 5.4e20    # fótons / (cm^2 * s) (baseado em 110 W/cm^2)

In [6]:
# 2. Sistema de Equações de Taxa
def sistema_edus(t, n, W51, W52, freq):
    n5, n2, n3 = n
    # Função de modulação do laser F(t) = 1 + mu * cos(omega * t)
    omega = 2 * np.pi * freq
    F_t = 1 + 0.1 * np.cos(omega * t) 
    
    # Derivadas (Assumindo n1, n4 constantes como no modelo do paper)
    dn5_dt = sigma * P * F_t - W51 * n5 - W52 * n5 - k54 * n5
    dn2_dt = W51 * n5 - W52 * n5 - k21 * n2
    dn3_dt = W52 * n5 - k31 * n3
    
    return [dn5_dt, dn2_dt, dn3_dt]


In [ ]:
# 3. Função de Custo para o Ajuste (Residuals)
def funcao_residual(parametros, freq_exp, fase_exp):
    W51, W52 = parametros
    fases_calc = []
    
    for f in freq_exp:
        # Integração do sistema para a frequência f
        t_span = (0, 1/f * 5) # Simula alguns ciclos
        sol = solve_ivp(sistema_edus, t_span, [0.1, 0, 0], args=(W51, W52, f))
        
        # Calcular fase teórica (exemplo simplificado)
        # Você deve extrair a fase da resposta de n3(t) usando FFT
        fase_calc = calcular_fase_via_fft(sol.t, sol.y[2])
        fases_calc.append(fase_calc)
        
    return np.array(fases_calc) - fase_exp

# 4. Execução do Ajuste
# Supõe que você tenha seus dados experimentais em 'freq_data' e 'phase_data'
initial_guess = [1e-15, 1e-15] # Palpites iniciais para W51, W52
resultado = least_squares(funcao_residual, initial_guess, args=(freq_data, phase_data))

print(f"Valores encontrados: W51 = {resultado.x[0]}, W52 = {resultado.x[1]}")